In [ ]:
#@title Prevent disconnections
%%html
<audio src="https://oobabooga.github.io/silence.m4a" controls>

In [ ]:
#@title Main Code
!apt install aria2

# Install cloudflared for the public tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Clone ComfyUI (everything stays in the Colab instance, no Drive)
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI

# Install dependencies (Colab already ships a CUDA-enabled torch)
%cd /content/ComfyUI
!pip install -r requirements.txt -q


In [ ]:
#@title Select Model
KEY = ""  # @param {type:"string"}
KEY = "token=" + KEY.strip()

MODELS = {
    "cyberrealisticPony_semiRealV40": {
        "file": "cyberrealisticPony_semiRealV40.safetensors",
        "url": f"https://civitai.com/api/download/models/2268768?type=Model&format=SafeTensor&size=pruned&fp=fp16&{KEY}",
    },
    "deepDarkHentaiMixNSFW_v61Hybrid": {
        "file": "deepDarkHentaiMixNSFW_v61Hybrid.safetensors",
        "url": f"https://civitai.com/api/download/models/634653?type=Model&format=SafeTensor&size=pruned&fp=fp16&{KEY}",
    },
    "novaCartoon_v10": {
        "file": "novaCartoon_v10.safetensors",
        "url": f"https://civitai.com/api/download/models/821389?type=Model&format=SafeTensor&size=pruned&fp=fp16&{KEY}",
    },
}

CHOICE = "cyberrealisticPony_semiRealV40"  # @param ["cyberrealisticPony_semiRealV40", "deepDarkHentaiMixNSFW_v61Hybrid", "novaCartoon_v10"]

MODEL = MODELS[CHOICE]
MODEL_PATH = "/content/ComfyUI/models/checkpoints"


In [ ]:
#@title Download Model
USER_AGENT = '"User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64)"'

!mkdir -p $MODEL_PATH
!aria2c --enable-http-keep-alive=false --header=$USER_AGENT --console-log-level=error -c -x 16 -s 16 -k 1M --summary-interval=5 -d $MODEL_PATH -o "{MODEL['file']}" "{MODEL['url']}"


In [ ]:
#@title LAUNCH!
# Start ComfyUI and expose it through a Cloudflare tunnel.
# Look for the "https://xxx.trycloudflare.com" URL in the output.
import subprocess, time, re, urllib.request, os

os.chdir("/content/ComfyUI")

# Start ComfyUI in the background
!nohup python main.py --listen 0.0.0.0 --port 8188 > /content/comfyui.log 2>&1 &

# Wait until the server is reachable
for _ in range(180):
    try:
        urllib.request.urlopen("http://localhost:8188", timeout=2)
        print("ComfyUI is up!")
        break
    except Exception:
        time.sleep(1)

# Open the tunnel and print the public URL
proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

for line in proc.stdout:
    line = line.strip()
    if line:
        print(line)
    if "trycloudflare.com" in line:
        print("\nComfyUI is ready, open the URL above in your browser.")
        break
